# Applet interactiva 

In [ ]:
%run melkman.ipynb

In [ ]:
def nombre(v, P):
    """
    Input: v (vértice), P (vértices del polígono)
    Output: nombre del vértice ('v0', 'v1', ...)
    """
    return "v%d" % P.index(v)

In [ ]:
def estado_previo(pasos, k):
    """
    Input: pasos(lista de melkman), k (nº de paso, debe ser k>=1)
    Output: C (cierre antes del paso k)
            q (último vértice añadido al cierre antes del paso k)
    
    En el paso k se comprueba si el vértice p está en R_q, usando el cierre y el q que había justo antes.
    La variable 'pasos', guarda el cierre DESPUÉS de cada paso, así que hay que mirar el paso anterior
    """
    C = pasos[k-1]['C']
    j = k-1
    while pasos[j]['caso'] == 1: # los descartados no cambian q
        j -= 1

    return C, pasos[j]['p']

In [ ]:
def region_Rq(q, C, O, L):
    """
    Input: q (vértice del cierre), C (cierre), O (orientación), L (longitud de los lados)
    Output: polígono grande que representa la región R_q
    
    R_q es el ángulo con vértice q, formado por las rectas que pasan por q y sus dos vecinos en C (ANTERIOR y POSTERIOR).
    Si un vértice cae ahí, no puede estar en el cierre (es el mismo caso que en_Rq)
    Si q y sus vecinos etán alineados, entonces R_q será un semiplano
    """

    # definimos a y b, vecinos de q en C
    a, b = ANTERIOR(q, C), POSTERIOR(q, C)
    ra = (q[0]+ L*(a[0]-q[0]), q[1] + L*(a[1]-q[1]))
    rb = (q[0]+ L*(b[0]-q[0]), q[1] + L*(b[1]-q[1]))
    if signo(a, q, b) != 0:
        return [q, ra, rb]

    # Caso alineado: rectángulo hacia el lado de R_q
    n = (-(b[1] - q[1]), b[0]-q[0]) # perpendicular a q: b
    if O*signo(q, b, (q[0]+n[0], q[1]+n[1])) <0:
        n = (-n[0], -n[1])
    return [ra, rb, (rb[0] + L*n[0], rb[1]+ L*n[1]), (ra[0]+L*n[0], ra[1]+ L*n[1])]


In [ ]:
def describir_paso(P, pasos, k):
    """
    Input: P(vértices del polígono), pasos (lista de melkman), k (nº de paso)
    Output: texto con el estado del algoritmo en el paso k (abordamos distintos casos)
    """

    N = len(pasos)
    paso = pasos[k]
    p = nombre(paso['p'], P)
    deque = lambda C: " ".join(nombre(v, P) for v in C)

    lineas = ["Paso %d de %d" % (k, N-1)]
    if paso['caso'] == 'inicio': # caso inicial
        lineas.append("Triángulo inicial con v0, v1 y v2")
    else:
        C_antes, q = estado_previo(pasos, k)
        lineas.append("Deque C antes:  " + deque(C_antes))
        lineas.append("q (último añadido):  " + nombre(q, P))
        if paso['caso'] == 1:
            lineas.append(p + " ∈ R_q -> se descarta (no puede estar en el cierre)")
        else:
            lineas.append(p + " ∉ R_q -> entra en el cierre")
            if paso['borrados']:
                lineas.append("Se eliminan: " + ", ".join(nombre(b, P) for b in paso['borrados']))

    lineas.append("Deque C después: " + deque(paso['C']))
    return "\n".join(lineas)

In [ ]:
def dibujar_paso(P, pasos, k):
    """
    Input: P (vértices del polígono), pasos (lista de melkman), k (nº de paso)
    Output: gráfico de Sage con el estado del algoritmo en el paso k

    Morado: región R_q con la que se comprueba el vértice del paso
    Gris claro: parte del polígono que aún no se ha leído
    Negro: parte ya recorrida
    Verde: cierre convexo después del paso
    Rojo/azul: vértice del paso (descartado/añadido)
    Naranja: vértices eliminados del cierre en este paso
    """
    paso = pasos[k]
    p, caso, C, borrados = paso['p'], paso['caso'], paso['C'], paso['borrados']
    i = P.index(p) # se han leído los vértices v0,...vi

    # Marco del dibujo y separación de las etiqeutas 
    xs = [v[0] for v in P]
    ys = [v[1] for v in P]
    tam = max(max(xs) - min(xs), max(ys)-min(ys))
    m, d = 0.1*tam, 0.03*tam 
    marco = dict(xmin=min(xs)-m, xmax=max(xs)+m, ymin=min(ys)-m, ymax=max(ys)+m)

    G = Graphics()
    if caso != 'inicio':
        C_antes, q = estado_previo(pasos, k)
        O = signo(P[0], P[1], P[2]) # orientación del triángulo inicial
        G += polygon(region_Rq(q, C_antes, O, 100), color='purple', alpha=0.15) #R_q
        G += point(q, color='purple', size=60)
    G += polygon(P, fill=False, color='lightgray', linestyle='--') # polígono completo
    G += line(P[:i+1], color='black', thickness=1.5) # parte recorrida
    G += polygon(C, color = 'green', alpha=0.25) # cierre actual
    G += line(C + [C[0]], color='green', thickness=2.5)
    G += points(P[i+1:], color='lightgray', size=25) # vértices sin leer
    G += points(P[:i+1], color='black', size=25) # vértices leídos

    if borrados:
        G+= points(borrados, color='orange', size=120, marker='x')
    color = {'inicio': 'purple', 1: 'red', 2: 'blue'}[caso]
    G += point(p, color=color, size=90)
    for j, v in enumerate(P):
        G += text("v%d" % j, (v[0] + d, v[1] + d), color='gray', fontsize=9)
    G.set_axes_range(**marco) # recorta R_q al marco del polígono 
    return G

In [ ]:
def areaSignada(a,b,c):
    """
    Input: a, b, c (puntos (x, y))
    Output: área con signo del triángulo a b c

    Práctica 1. Es la mitad de signo(a, b, c), así que tiene el mismo signo:
        si > 0 giro a la izquierda (antihorario)
        si < 0 giro a la derecha (horario)
        si = 0 alineados
    """
    return((b[0]-a[0])*(c[1]-a[1])-(b[1]-a[1])*(c[0]-a[0]))/2


def enSegmentoII(p,s):
    """
    Input: p (punto), s (segmento [a, b])
    Output: True si p está en el segmento s, False si no

    Práctica 1. p está en s si está alineado con sus extremos y su x y su y están
    entre las de los extremos (hay que mirar las dos, por si el segmento es vertical).
    Usamos esta versión y no enSegmento porque no calcula raíces cuadradas:
    con decimales, comparar distancias con == puede fallar por el redondeo
    """
    if areaSignada(p,s[0],s[1])!=0:
        return False
    if s[0][0]<=p[0] and p[0]<=s[1][0] or s[0][0]>=p[0] and p[0]>=s[1][0]:
        if s[0][1]<=p[1] and p[1]<=s[1][1] or s[0][1]>=p[1] and p[1]>=s[1][1]:
            return True
    return False


def testInterseccionSegmentos(p,q):
    """
    Input: p, q (segmentos [a, b])
    Output: True si los segmentos se cortan o se tocan, False si no

    Práctica 1. Se cortan si los extremos de q quedan a distinto lado de p y los extremos
    de p a distinto lado de q (áreas signadas de signo opuesto). Si no, puede que se toquen:
    se mira si algún extremo está sobre el otro segmento
    """
    if (areaSignada(p[0],p[1],q[0])*areaSignada(p[0],p[1],q[1])<0 and areaSignada(q[0],q[1],p[0])*areaSignada(q[0],q[1],p[1])<0):
        return True
    if enSegmentoII(p[0],q) or enSegmentoII(p[1],q) or enSegmentoII(q[0],p) or enSegmentoII(q[1],p):
        return True
    return False


def testPoligonoSimple(P):
    """
    Input: P (vértices del polígono, en orden)
    Output: True si el polígono es simple, False si dos de sus lados se cortan

    Práctica 4. Compara cada lado con todos los lados que no son vecinos suyos
    (los vecinos comparten un vértice y siempre se tocan)
    """
    n = len(P)
    for i in range(n):
        for j in range(i+2, n): # j = i+1 es el lado vecino: se salta
            if i == 0 and j == n-1: # el primer y el último lado también son vecinos
                continue
            if testInterseccionSegmentos([P[i], P[(i+1)%n]], [P[j], P[(j+1)%n]]):
                return False
    return True

In [ ]:
import json

def datos_reproductor(nombre_pol, P, casco, pasos):
    """
    Input: nombre_pol (nombre del polígono), P (vértices), casco y pasos (salida de melkman)
    Output: texto JSON con todo lo que necesita el reproductor de la web

    Los vértices se guardan por su índice (0 = v0, 1 = v1, ...). Para cada paso se añade
    el cierre antes del paso, q y la región R_q, calculados con estado_previo y region_Rq
    """
    indice = {v: i for i, v in enumerate(P)}
    idx = lambda lista: [int(indice[v]) for v in lista]
    O = signo(*pasos[0]['C']) # orientación del triángulo inicial
    lista = []
    for k, paso in enumerate(pasos):
        caso = paso['caso'] if paso['caso'] == 'inicio' else int(paso['caso'])
        d = {'p': int(indice[paso['p']]), 'caso': caso,
             'C': idx(paso['C']), 'borrados': idx(paso['borrados'])}
        if k > 0:
            C_antes, q = estado_previo(pasos, k)
            d['C_antes'] = idx(C_antes)
            d['q'] = int(indice[q])
            d['Rq'] = [[float(x), float(y)] for (x, y) in region_Rq(q, C_antes, O, 100)]
        lista.append(d)
    return json.dumps({'nombre': str(nombre_pol),
                       'P': [[float(x), float(y)] for (x, y) in P],
                       'casco': idx(casco), 'pasos': lista})


def enviar_a_la_web(datos):
    """
    Input: datos (texto JSON)
    Output: ninguno; muestra en la celda el bloque que lee el reproductor de la web
    """
    datos = datos.replace('</', '<\\/') 
    show(html('<script type="application/json" class="mk-data">%s</script>'
              '<script>window.MelkmanPlayer && window.MelkmanPlayer.scan();</script>' % datos))


@interact
def applet(poligono=selector(list(ejemplos.keys()) + ['Personalizado'],
                             label='Polígono / Polygon', buttons=False),
           coords=input_box(default=[(0,0), (4,0), (4,3), (2,1), (0,3)],
                            label='Coordenadas / Coordinates', width=60)):

    # 1. Polígono elegido 
    P = coords if poligono == 'Personalizado' else ejemplos[poligono]
    P = [(float(x), float(y)) for (x, y) in P]

    # 2. Comprobaciones: si algo falla, la web muestra el aviso
    error = None
    if len(P) < 3:
        error = 'pocos'
    elif len(set(P)) < len(P):
        error = 'repetidos'
    elif not testPoligonoSimple(P):      
        error = 'no_simple'
    if error is None:
        try:
            casco, pasos = melkman(P) # 3. Algoritmo de Melkman 
        except ValueError:
            error = 'alineados'
    if error is not None:
        enviar_a_la_web(json.dumps({'error': error, 'nombre': str(poligono)}))
        return

    # 4. El reproductor de la web dibuja los pasos que ha calculado SageMath
    print('SageMath: melkman() ha procesado %d vértices en %d pasos; el cierre tiene %d vértices.'
          % (len(P), len(pasos), len(casco)))
    enviar_a_la_web(datos_reproductor(poligono, P, casco, pasos))